# MLAT Feature Research

A separate, versioned experiment that consumes trusted artifacts from
`statistical_feature_research.ipynb`. It does not rewrite that historical
research cycle, inspect MGC transfer results, use the exposed Final-test
outcomes, construct a trading signal, or run a strategy backtest.

## 0.0 MLAT Research Mandate and Governance

Test whether a pre-registered batch of 12 book-derived or book-motivated
causal features adds stable information beyond the existing one-minute GC
feature matrix. Direction, expansion, volatility, and path-risk evidence are
evaluated separately under the frozen v1 contract. Honest rejection is a
valid result.

## 1.0 Environment and Reproducibility

### 1.1 Imports and Versions

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import subprocess
import sys
import time
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import psutil
import pyarrow
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import scipy

REQUIRED = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "nbformat": nbformat.__version__,
}
REQUIRED

{'numpy': '2.2.3',
 'pandas': '3.0.3',
 'pyarrow': '25.0.0',
 'scipy': '1.17.1',
 'matplotlib': '3.11.1',
 'nbformat': '5.10.4'}

### 1.2 Paths and Configuration

In [2]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / ".git").exists():
            return candidate
    raise RuntimeError(f"Repository root not found from {start}")


ROOT = find_repo_root(Path.cwd())
SOURCE_ROOT = ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from statistical_research.mlat_artifacts import (
    MLATArtifactPaths,
    prepare_csv_table,
    save_csv,
    save_json,
    save_parquet,
    verify_saved_table,
)
from statistical_research.mlat_feature_engineering import (
    BAR_INPUT_COLUMNS,
    FEATURE_METADATA_COLUMNS,
    MLAT_COMPLETE_HISTORY_BARS,
    build_mlat_feature_matrix,
)
from statistical_research.mlat_feature_evaluation import (
    MlatEvaluationConfig,
    assign_mlat_verdicts,
    build_mlat_evaluation_frame,
    evaluate_incremental_information,
    evaluate_mlat_feature_cells,
)
from statistical_research.mlat_feature_registry import (
    MLAT_FEATURE_NAMES,
    build_mlat_registry,
    registry_to_frame,
    validate_mlat_registry,
)
from statistical_research.mlat_feature_validation import (
    audit_feature_overlap,
    validate_mlat_feature_matrix,
    verify_reloaded_feature_matrix,
)
from statistical_research.mlat_volatility_models import audit_segmented_garch

PATHS = MLATArtifactPaths(ROOT, version="v1").ensure()
DOCS = ROOT / "project_docs" / "mlat_feature_research"
UPSTREAM = {
    "bars": ROOT / "data" / "processed" / "research_bars_gc_mgc_1m.parquet",
    "eligible": ROOT / "data" / "processed" / "statistical_research" / "eligible_observations_gc.parquet",
    "labels": ROOT / "data" / "processed" / "statistical_research" / "forward_labels_gc.parquet",
    "features": ROOT / "data" / "processed" / "statistical_research" / "feature_matrix_gc.parquet",
    "registry": ROOT / "data" / "processed" / "statistical_research" / "feature_registry_gc.parquet",
    "frozen": ROOT / "data" / "processed" / "statistical_research" / "frozen_expansion_feature_set_gc.parquet",
}
assert all(path.exists() for path in UPSTREAM.values())

### 1.3 Random Seeds

In [3]:
SEED = 20260723
HORIZONS = (5, 15, 30, 60, 120, 180)
PARTITIONS = ("Development", "Validation")
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
warnings.simplefilter("always")

RUN_STARTED = time.perf_counter()
PROCESS = psutil.Process()
START_RSS_BYTES = PROCESS.memory_info().rss

### 1.4 Git and Artifact Metadata

In [4]:
def git(*args: str) -> str:
    result = subprocess.run(
        ["git", *args],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()


RUNTIME_IDENTITY = {
    "python": sys.version,
    "platform": platform.platform(),
    "git_commit": git("rev-parse", "HEAD"),
    "git_branch": git("branch", "--show-current"),
    "seed": SEED,
    "dependencies": REQUIRED,
}
RUNTIME_IDENTITY

{'python': '3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]',
 'platform': 'Windows-11-10.0.26200-SP0',
 'git_commit': 'e75cab144f9f2096d51262c5da478166e116c9c5',
 'git_branch': 'twatski',
 'seed': 20260723,
 'dependencies': {'numpy': '2.2.3',
  'pandas': '3.0.3',
  'pyarrow': '25.0.0',
  'scipy': '1.17.1',
  'matplotlib': '3.11.1',
  'nbformat': '5.10.4'}}

## 2.0 Upstream Research Artifact Contract

### 2.1 Load Trusted GC Bars

In [5]:
def parquet_frame(
    path: Path,
    columns: list[str] | tuple[str, ...],
    expression=None,
) -> pd.DataFrame:
    dataset = ds.dataset(path, format="parquet")
    table = dataset.to_table(columns=list(columns), filter=expression)
    return table.to_pandas(ignore_metadata=True)

def gc_bar_frame_with_source_ids(path: Path) -> pd.DataFrame:
    physical_columns = [
        column for column in BAR_INPUT_COLUMNS if column != "source_row_id"
    ]
    parquet_file = pq.ParquetFile(path)
    frames = []
    physical_offset = 0
    for batch in parquet_file.iter_batches(
        batch_size=250_000,
        columns=physical_columns,
    ):
        frame = batch.to_pandas()
        source_ids = np.arange(
            physical_offset,
            physical_offset + len(frame),
            dtype=np.int64,
        )
        physical_offset += len(frame)
        gc_mask = frame["product"].eq("GC").to_numpy()
        if gc_mask.any():
            gc_frame = frame.loc[gc_mask].copy()
            gc_frame.insert(0, "source_row_id", source_ids[gc_mask])
            frames.append(gc_frame)
    result = pd.concat(frames, ignore_index=True)
    return result.loc[:, BAR_INPUT_COLUMNS]


bars = gc_bar_frame_with_source_ids(UPSTREAM["bars"])
assert bars["product"].eq("GC").all()
assert tuple(bars.columns) == tuple(BAR_INPUT_COLUMNS)
bars.shape, bars["ts_event_utc"].min(), bars["ts_event_utc"].max()

((1759671, 17),
 Timestamp('2021-05-24 00:00:00+0000', tz='UTC'),
 Timestamp('2026-05-22 20:59:00+0000', tz='UTC'))

### 2.2 Load Eligible Observations

In [6]:
eligible_columns = [
    "observation_id", "decision_bar_id", "decision_timestamp_utc",
    "decision_timestamp_ny", "entry_timestamp_utc", "entry_timestamp_ny",
    "trade_date_ny", "entry_session", "research_partition", "product",
    "symbol", "active_symbol", "instrument_id", "continuous_segment_id",
]
dv_filter = ds.field("research_partition").isin(list(PARTITIONS))
eligible = parquet_frame(UPSTREAM["eligible"], eligible_columns, dv_filter)
assert set(eligible["research_partition"]) == set(PARTITIONS)
assert eligible["product"].eq("GC").all()
eligible["research_partition"].value_counts()

research_partition
Development    306230
Validation     118076
Final test          0
Name: count, dtype: int64

### 2.3 Load Forward Labels

In [7]:
label_columns = [
    "observation_id", "decision_bar_id", "decision_timestamp_utc",
    "decision_timestamp_ny", "entry_timestamp_utc",
    "entry_timestamp_ny", "trade_date_ny", "entry_session",
    "research_partition", "product", "symbol", "active_symbol",
    "instrument_id", "continuous_segment_id", "decision_atr_20m",
    "atr_normalization_available",
]
for horizon in HORIZONS:
    label_columns.extend([
        f"forward_return_{horizon}_ticks",
        f"forward_return_{horizon}_atr",
        f"future_range_{horizon}_atr",
        f"future_realized_volatility_{horizon}_bps",
        f"mae_long_{horizon}_atr",
        f"mae_short_{horizon}_atr",
        f"label_available_{horizon}",
    ])
labels = parquet_frame(UPSTREAM["labels"], label_columns, dv_filter)
assert set(labels["research_partition"]) == set(PARTITIONS)
assert labels["product"].eq("GC").all()
labels.shape

(424306, 58)

### 2.4 Load Existing Feature Registry and Matrix

In [8]:
existing_registry = pd.read_parquet(UPSTREAM["registry"])
existing_names = existing_registry["feature_name"].astype(str).tolist()
existing_columns = list(FEATURE_METADATA_COLUMNS) + existing_names
existing_features = parquet_frame(UPSTREAM["features"], existing_columns, dv_filter)
assert existing_features["product"].eq("GC").all()
existing_features.shape, existing_registry.shape

((424306, 97), (85, 24))

### 2.5 Load Baseline and Expansion Anchors

In [9]:
frozen_expansion = pd.read_parquet(UPSTREAM["frozen"])
frozen_anchor_names = frozen_expansion.loc[
    frozen_expansion["in_frozen_set"].astype(bool), "feature_name"
].astype(str).tolist()
assert len(frozen_anchor_names) == 15
assert "atr_20" in frozen_anchor_names
frozen_expansion.loc[frozen_expansion["in_frozen_set"].astype(bool)]

,feature_name,cluster_id,is_experimental,role,in_frozen_set,decision_reason,selection_score
0,atr_20,19,False,anchor,True,anchor_representative,0.547765
1,minute_from_execution_window_open,20,False,representative,True,confirmed_incremental_information,0.442712
2,minutes_to_1530_forced_exit,21,False,representative,True,confirmed_incremental_information,0.442712
3,atr_ratio_20_60,18,False,representative,True,confirmed_incremental_information,0.356985
4,volume_acceleration_5_20,27,False,representative,True,confirmed_incremental_information,0.140237
5,efficiency_ratio_60,17,False,representative,True,confirmed_incremental_information,0.138860
6,return_sign_change_rate_30,9,False,representative,True,confirmed_incremental_information,0.134272
7,ols_r_squared_30,16,False,representative,True,confirmed_incremental_information,0.133256
8,atr_ratio_5_20,26,False,representative,True,confirmed_incremental_information,0.132511
9,efficiency_ratio_30,15,False,representative,True,confirmed_incremental_information,0.128899


### 2.6 Validate Upstream Artifact Integrity

In [10]:
upstream_manifest = pd.read_csv(DOCS / "mlat_upstream_artifact_manifest.csv")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


upstream_check_rows = []
for logical_name, path in UPSTREAM.items():
    relative = path.relative_to(ROOT).as_posix()
    manifest_row = upstream_manifest.loc[
        upstream_manifest["exact_path"].astype(str).eq(relative)
    ]
    assert len(manifest_row) == 1, relative
    expected = manifest_row.iloc[0]
    metadata = pq.ParquetFile(path).metadata
    observed_hash = sha256_file(path)
    upstream_check_rows.append({
        "artifact": logical_name,
        "exact_path": relative,
        "exists": path.is_file(),
        "row_count_pass": metadata.num_rows == int(expected["row_count"]),
        "column_count_pass": metadata.num_columns == int(expected["column_count"]),
        "sha256_pass": observed_hash == str(expected["file_sha256"]),
        "schema_fingerprint_recorded": bool(str(expected["schema_fingerprint"])),
        "manifest_validation_status": str(expected["validation_status"]),
    })
upstream_checks = pd.DataFrame(upstream_check_rows)
assert upstream_checks[[
    "exists", "row_count_pass", "column_count_pass", "sha256_pass",
    "schema_fingerprint_recorded",
]].all().all()
assert upstream_checks["manifest_validation_status"].str.startswith(
    "VALIDATED"
).all()
assert eligible["observation_id"].is_unique
assert labels["observation_id"].is_unique
assert existing_features["observation_id"].is_unique
upstream_checks

,artifact,exact_path,exists,row_count_pass,column_count_pass,sha256_pass,schema_fingerprint_recorded,manifest_validation_status
0,bars,data/processed/research_bars_gc_mgc_1m.parquet,True,True,True,True,True,VALIDATED_WITH_COLUMN_WHITELIST
1,eligible,data/processed/statistical_research/eligible_o...,True,True,True,True,True,VALIDATED
2,labels,data/processed/statistical_research/forward_la...,True,True,True,True,True,VALIDATED_DEV_VALIDATION_ONLY
3,features,data/processed/statistical_research/feature_ma...,True,True,True,True,True,VALIDATED
4,registry,data/processed/statistical_research/feature_re...,True,True,True,True,True,VALIDATED
5,frozen,data/processed/statistical_research/frozen_exp...,True,True,True,True,True,VALIDATED


## 3.0 MLAT Book Traceability

### 3.1 Book Coverage Status

In [11]:
stage1_report = (DOCS / "mlat_stage1_completion_report.md").read_text(encoding="utf-8")
assert "Status: COMPLETE" in stage1_report
coverage_manifest = pd.read_csv(DOCS / "mlat_book_ingestion_manifest.csv")
assert coverage_manifest["text_extraction_status"].eq("COMPLETE").all()
coverage_manifest.groupby(["section_type", "status"]).size()

section_type  status  
appendix      COMPLETE     1
chapter       COMPLETE    23
contents      COMPLETE     1
front_matter  COMPLETE     1
index         COMPLETE     1
preface       COMPLETE     1
references    COMPLETE     1
dtype: int64

### 3.2 Relevant Concepts

In [12]:
concept_registry = pd.read_csv(DOCS / "mlat_concept_registry.csv")
formula_text = (DOCS / "mlat_formulas_and_definitions.md").read_text(encoding="utf-8")
concept_registry.shape, formula_text.count("## MLAT-F")

((30, 15), 14)

### 3.3 Project Applicability

In [13]:
hypothesis_text = (
    DOCS / "mlat_feature_hypothesis_catalog.md"
).read_text(encoding="utf-8")
hypothesis_rows = []
for line in hypothesis_text.splitlines():
    if line.startswith("| MLAT-H"):
        values = [value.strip() for value in line.strip("|").split("|")]
        if len(values) == 9:
            hypothesis_rows.append(values)
hypothesis_catalog = pd.DataFrame(
    hypothesis_rows,
    columns=[
        "hypothesis_id", "feature_name", "source", "source_type",
        "target", "overlap", "leakage", "cost", "decision",
    ],
)
assert len(hypothesis_catalog) == 29
hypothesis_catalog["decision"].value_counts(dropna=False)

decision
SELECT_V1               12
REJECT                   7
LATER_PHASE              4
DEFER                    2
DEFER_DIFFERENT_DATA     2
AUDIT_ONLY               1
REJECT_GOVERNANCE        1
Name: count, dtype: int64

### 3.4 Frozen Feature Batch

In [14]:
source_traceability = pd.read_csv(DOCS / "mlat_source_traceability.csv")
assert set(MLAT_FEATURE_NAMES).issubset(set(source_traceability["feature_or_method"]))
source_traceability

,hypothesis_id,feature_or_method,source_chapter,source_pdf_page,book_concept,project_adaptation,implementation_target,status
0,MLAT-H001,bollinger_zscore_20,4; Appendix,131-133; 740-742,Bollinger mean/bands and examples,Adapted standardized z-score,mlat_feature_engineering.py,FROZEN
1,MLAT-H002,bollinger_bandwidth_20,Appendix,740-742,Normalized band width/squeeze,Direct formula adaptation,mlat_feature_engineering.py,FROZEN
2,MLAT-H003,cutler_rsi_14,4,132,RSI with 14-period example,Cutler rolling convention declared by project,mlat_feature_engineering.py,FROZEN
3,MLAT-H004,chaikin_money_flow_20,Appendix,752-753,Money-flow multiplier and volume,Rolling normalized adaptation,mlat_feature_engineering.py,FROZEN
4,MLAT-H005,amihud_illiquidity_60,20; Appendix,656; 752,Absolute return / dollar volume rolling measure,Within-GC close-volume proxy,mlat_feature_engineering.py,FROZEN
5,MLAT-H006,parkinson_volatility_30,9,297-301,Volatility modelling motivation and anchor com...,Project-original estimator; formula not claime...,mlat_feature_engineering.py,FROZEN
6,MLAT-H007,rogers_satchell_volatility_30,9,297-301,Volatility modelling motivation and anchor com...,Project-original estimator; formula not claime...,mlat_feature_engineering.py,FROZEN
7,MLAT-H008,realized_semivariance_balance_60,5; 9,169-178; 297-301,Downside risk and changing variance,Project-original signed variation state,mlat_feature_engineering.py,FROZEN
8,MLAT-H009,bipower_jump_ratio_60,9,297-301,Changing variance and residual diagnostics,Project-original jump-state estimator,mlat_feature_engineering.py,FROZEN
9,MLAT-H010,variance_ratio_60_5,9,280-296,"Stationarity, AR dependence, forecast diagnostics",Adapted local random-walk diagnostic,mlat_feature_engineering.py,FROZEN


### 3.5 Frozen Evaluation Contract

In [15]:
applicability_text = (
    DOCS / "mlat_project_applicability_matrix.md"
).read_text(encoding="utf-8")
rejected_text = (
    DOCS / "mlat_feature_batch_v1.md"
).read_text(encoding="utf-8")
print(applicability_text.splitlines()[0])
print(rejected_text.split("## Explicit exclusions", 1)[1][:1_500])

# MLAT Project Applicability Matrix


- GARCH is an audit-only method, not a v1 matrix feature. The prior
  implementation fails GC isolation and boundary governance.
- Kalman/KAMA are deferred because their fitted/recursive state and overlap
  are not justified for the first batch.
- Wavelet denoising is rejected until a genuinely one-sided implementation
  is defined.
- PPO/MACD, NATR, Williams %R, raw AR(1), OBV, and clock features are
  rejected as redundant or reset-ambiguous.
- Cross-sectional, text, quote/trade/order-book, macro, image, deep, GAN,
  and reinforcement-learning methods require a different phase or data.

No membership or parameter may change after Validation results are viewed.



## 4.0 Existing-Feature Overlap Audit

### 4.1 Formula and Definition Comparison

In [16]:
existing_registry[[
    column for column in
    ["feature_name", "feature_family", "formula_or_definition", "target_family"]
    if column in existing_registry.columns
]].head()

,feature_name,formula_or_definition
0,return_1m_bps,"10,000 × log(close_t / close_(t-1))"
1,return_5m_bps,"10,000 × log(close_t / close_(t-5))"
2,return_15m_bps,"10,000 × log(close_t / close_(t-15))"
3,return_30m_bps,"10,000 × log(close_t / close_(t-30))"
4,return_60m_bps,"10,000 × log(close_t / close_(t-60))"


### 4.2 Exact Duplicate Detection

In [17]:
preimplementation_overlap = pd.read_csv(
    DOCS / "mlat_existing_feature_overlap_audit.csv"
)
preimplementation_overlap

,hypothesis_id,feature_name,nearest_existing_features,formula_overlap_assessment,preimplementation_redundancy_risk,preimplementation_decision,observed_nearest_feature,observed_max_abs_spearman_development,observed_exact_duplicate,final_eligibility
0,MLAT-H001,bollinger_zscore_20,rolling_range_position_15; normalized_ols_slop...,related but not formula-equivalent,MEDIUM,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
1,MLAT-H002,bollinger_bandwidth_20,range_compression_ratio_5_30; atr_ratio_20_60,same broad compression target; different close...,HIGH,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
2,MLAT-H003,cutler_rsi_14,return_15m_bps; directional_persistence_15,bounded gain/loss transform is distinct,MEDIUM,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
3,MLAT-H004,chaikin_money_flow_20,close_location_value; signed_volume_proxy; vol...,interaction/aggregation is distinct but likely...,HIGH,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
4,MLAT-H005,amihud_illiquidity_60,volume_per_tick_range; liquidity_vacuum_score_exp,inverse activity-impact form is distinct,HIGH,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
5,MLAT-H006,parkinson_volatility_30,atr_20; realized_volatility_30; current_range_...,high-low quadratic estimator is distinct,HIGH,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
6,MLAT-H007,rogers_satchell_volatility_30,atr_20; realized_volatility_30,OHLC drift-robust estimator is distinct,HIGH,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
7,MLAT-H008,realized_semivariance_balance_60,directional_energy_balance_15_exp,related signed-energy idea at a different defi...,HIGH,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
8,MLAT-H009,bipower_jump_ratio_60,current_range_over_atr; liquidity_vacuum_score...,jump share is distinct,MEDIUM,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN
9,MLAT-H010,variance_ratio_60_5,return_autocorrelation_15; return_sign_change_...,same persistence family but non-equivalent var...,HIGH,ELIGIBLE_PENDING_EMPIRICAL,NaN,NaN,NaN,NaN


### 4.3 Correlation and Monotonic-Transform Checks

In [18]:
frozen_anchor_names

['atr_20',
 'minute_from_execution_window_open',
 'minutes_to_1530_forced_exit',
 'atr_ratio_20_60',
 'volume_acceleration_5_20',
 'efficiency_ratio_60',
 'return_sign_change_rate_30',
 'ols_r_squared_30',
 'atr_ratio_5_20',
 'efficiency_ratio_30',
 'relative_volume_20',
 'vwap_elasticity_30_exp',
 'current_range_over_atr',
 'choppiness_14',
 'efficiency_ratio_15']

### 4.4 Final Implementation Eligibility

In [19]:
batch_bytes = (DOCS / "mlat_feature_batch_v1.md").read_bytes()
contract_bytes = (
    DOCS / "mlat_feature_research_contract_v1.md"
).read_bytes()
BATCH_SHA256 = hashlib.sha256(batch_bytes).hexdigest()
CONTRACT_SHA256 = hashlib.sha256(contract_bytes).hexdigest()
assert BATCH_SHA256 == "8c8b74267635566f07011c2789ad4d323647f19eba1f1f3426a154c1a850acc0"
assert CONTRACT_SHA256 == "31ea8ff8255285d580dc9c2cb71bcbff85d05011e771b49f2f09d24cad810320"
{"batch_sha256": BATCH_SHA256, "contract_sha256": CONTRACT_SHA256}

{'batch_sha256': '8c8b74267635566f07011c2789ad4d323647f19eba1f1f3426a154c1a850acc0',
 'contract_sha256': '31ea8ff8255285d580dc9c2cb71bcbff85d05011e771b49f2f09d24cad810320'}

## 5.0 MLAT Feature Registry

### 5.1 Registry Schema

In [20]:
registry = build_mlat_registry()
registry_frame = registry_to_frame(registry)
validate_mlat_registry(registry_frame)
REGISTRY_VALID = True
registry_frame

,feature_name,display_name,feature_family,feature_tier,hypothesis_id,formula_id,source_chapter,source_pdf_page,source_type,formula_or_definition,...,missing_value_policy,normalization_method,output_dtype,existing_feature_overlap,leakage_risk,implementation_version,is_experimental,status,validation_minimum,validation_maximum
0,bollinger_zscore_20,Bollinger z-score (20),price_location,frozen_v1,MLAT-H001,MLAT-F002,4 / Appendix,131-133; 740-742,MLAT-ADAPTED,(close_t - population_mean_20(close)) / popula...,...,Null until 20 complete bars; zero standard dev...,Trailing 20-bar population mean and population...,float32,rolling_range_position_15; normalized_ols_slop...,LOW,1.0.0,True,FROZEN_V1,NaN,NaN
1,bollinger_bandwidth_20,Bollinger bandwidth (20),volatility_expansion,frozen_v1,MLAT-H002,MLAT-F003,Appendix,740-742,MLAT-DIRECT,4 * population_std_20(close) / population_mean...,...,Null until 20 complete bars; zero or non-posit...,Four-sigma width divided by trailing 20-bar me...,float32,range_compression_ratio_5_30; atr_ratio_5_20; ...,LOW,1.0.0,True,FROZEN_V1,0.0,NaN
2,cutler_rsi_14,Cutler RSI (14 changes),momentum_state,frozen_v1,MLAT-H003,MLAT-F004,4,132,MLAT-ADAPTED,"100 * sum_14(max(delta_close, 0)) / (sum_14(ma...",...,Null until 14 complete changes; zero gains-plu...,Simple rolling Cutler ratio; no Wilder recursion,float32,return momentum; directional_persistence_15,LOW,1.0.0,True,FROZEN_V1,0.0,100.0
3,chaikin_money_flow_20,Chaikin money flow (20),volume_price_state,frozen_v1,MLAT-H004,MLAT-F005,Appendix,752-753,MLAT-ADAPTED,sum_20(volume * ((2*close-high-low)/(high-low)...,...,Null until 20 complete bars; zero-range bars c...,Rolling money-flow volume divided by rolling v...,float32,close_location_value; signed_volume_proxy; vol...,LOW,1.0.0,True,FROZEN_V1,-1.0,1.0
4,amihud_illiquidity_60,Amihud illiquidity adaptation (60 returns),liquidity_risk,frozen_v1,MLAT-H005,MLAT-F006,20 / Appendix,656; 752,MLAT-ADAPTED,1e9 * mean_60(abs(log_return) / (close * volume)),...,Null until 60 complete returns; zero volume or...,One-billion-scaled mean inverse notional-activ...,float32,volume_per_tick_range; liquidity_vacuum_score_exp,LOW,1.0.0,True,FROZEN_V1,0.0,NaN
5,parkinson_volatility_30,Parkinson range volatility (30),range_volatility,frozen_v1,MLAT-H006,MLAT-F007,9,297-301,PROJECT-ORIGINAL-EXTENSION,1e4 * sqrt(mean_30(log(high/low)^2) / (4*log(2))),...,Null until 30 complete bars; non-positive high...,Unannualized one-minute estimator in basis points,float32,ATR; realized volatility; bar range,LOW,1.0.0,True,FROZEN_V1,0.0,NaN
6,rogers_satchell_volatility_30,Rogers-Satchell volatility (30),range_volatility,frozen_v1,MLAT-H007,MLAT-F008,9,297-301,PROJECT-ORIGINAL-EXTENSION,1e4 * sqrt(mean_30(log(high/open)*log(high/clo...,...,Null until 30 complete bars; non-positive OHLC...,Unannualized one-minute estimator in basis points,float32,ATR; realized volatility; bar range,LOW,1.0.0,True,FROZEN_V1,0.0,NaN
7,realized_semivariance_balance_60,Realized semivariance balance (60 returns),return_asymmetry,frozen_v1,MLAT-H008,MLAT-F009,5 / 9,169-178; 297-301,PROJECT-ORIGINAL-EXTENSION,(sum_60(r^2 * 1[r>0]) - sum_60(r^2 * 1[r<0])) ...,...,Null until 60 complete returns; zero total var...,Signed semivariance difference divided by tota...,float32,directional_energy_balance_15_exp,LOW,1.0.0,True,FROZEN_V1,-1.0,1.0
8,bipower_jump_ratio_60,Bipower jump ratio (60 products),jump_risk,frozen_v1,MLAT-H009,MLAT-F010,9,297-301,PROJECT-ORIGINAL-EXTENSION,"max(RV-BV, 0)/RV; RV=sum_60(r^2); BV=(pi/2)*(6...",...,Null until 61 complete returns; zero RV is nul...,Positive RV-minus-bipower share of realized va...,float32,current_range_over_atr; liquidity_vacuum_score...,LOW,1.0.0,True,FROZEN_V1,0.0,1.0
9,variance_ratio_60_5,Variance ratio (60 overlapping 5-return sums),serial_dependence,frozen_v1,MLAT-H010,MLAT-F011,9,280-296,MLAT-ADAPTED,population_var_60(sum_5(log_return)) / (5*popu...,...,Null until 60 complete overlapping 5-return su...,Five-return variance divided

### 5.2 Frozen Feature Definitions

In [21]:
assert registry_frame["feature_name"].tolist() == list(MLAT_FEATURE_NAMES)
registry_frame.dtypes.astype(str).to_frame("dtype")

,dtype
feature_name,str
display_name,str
feature_family,str
feature_tier,str
hypothesis_id,str
formula_id,str
source_chapter,str
source_pdf_page,str
source_type,str
formula_or_definition,str


### 5.3 Source and Page Traceability

In [22]:
registry_frame[[
    "feature_name", "minimum_history", "availability_timestamp",
    "reset_boundary", "missing_value_policy", "fitted_parameters",
]]

,feature_name,minimum_history,availability_timestamp,reset_boundary,missing_value_policy,fitted_parameters
0,bollinger_zscore_20,20,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 20 complete bars; zero standard dev...,None
1,bollinger_bandwidth_20,20,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 20 complete bars; zero or non-posit...,None
2,cutler_rsi_14,15,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 14 complete changes; zero gains-plu...,None
3,chaikin_money_flow_20,20,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 20 complete bars; zero-range bars c...,None
4,amihud_illiquidity_60,61,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 60 complete returns; zero volume or...,None
5,parkinson_volatility_30,30,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 30 complete bars; non-positive high...,None
6,rogers_satchell_volatility_30,30,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 30 complete bars; non-positive OHLC...,None
7,realized_semivariance_balance_60,61,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 60 complete returns; zero total var...,None
8,bipower_jump_ratio_60,62,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 61 complete returns; zero RV is nul...,None
9,variance_ratio_60_5,65,Close of completed decision bar t,"Reset at missing-minute, product, selected-con...",Null until 60 complete overlapping 5-return su...,None


### 5.4 Causal Availability Contract

In [23]:
registry_path = PATHS.data_path("feature_registry_mlat_gc")
registry_manifest_entry = save_parquet(
    registry_frame,
    registry_path,
    id_columns=("feature_name",),
    manifest_path=PATHS.manifest_path,
)
registry_save_checks = verify_saved_table(
    registry_frame,
    registry_path,
    id_columns=("feature_name",),
    manifest_entry=registry_manifest_entry,
)
registry_save_checks

,check_name,passed,expected,actual
0,file_exists,True,True,True
1,sha256,True,ad499707991d6b497ad94a3066fb915ba2885e2b8b9101...,ad499707991d6b497ad94a3066fb915ba2885e2b8b9101...
2,byte_size,True,25631,25631
3,row_count,True,12,12
4,column_count,True,31,31
5,column_order,True,"['feature_name', 'display_name', 'feature_fami...","['feature_name', 'display_name', 'feature_fami..."
6,pandas_dtypes,True,"{'feature_name': 'str', 'display_name': 'str',...","{'feature_name': 'str', 'display_name': 'str',..."
7,null_counts,True,"{'feature_name': 0, 'display_name': 0, 'featur...","{'feature_name': 0, 'display_name': 0, 'featur..."
8,id_values_and_order,True,exact,checked=['feature_name']
9,timestamps_exact,True,exact,checked=[]


## 6.0 MLAT Feature Engineering

### 6.1 Shared Causal Primitives

In [24]:
forbidden_tokens = (
    "forward_", "future_", "mfe_", "mae_", "label", "target",
    "entry_price", "stop_", "expansion_",
)
assert not any(
    token in column.lower()
    for column in BAR_INPUT_COLUMNS
    for token in forbidden_tokens
)
BAR_INPUT_COLUMNS

('source_row_id',
 'ts_event_utc',
 'ts_event_ny',
 'trade_date_ny',
 'product',
 'symbol',
 'active_symbol',
 'instrument_id',
 'continuous_segment_id',
 'open',
 'high',
 'low',
 'close',
 'volume',
 'tradable_research_flag',
 'roll_window_flag',
 'low_liquidity_warning_flag')

### 6.2 Return and Market-State Features

In [25]:
continuity_counts = bars[[
    "symbol", "active_symbol", "instrument_id", "continuous_segment_id",
    "tradable_research_flag", "roll_window_flag", "low_liquidity_warning_flag",
]].nunique(dropna=False)
continuity_counts

symbol                          26
active_symbol                   26
instrument_id                   26
continuous_segment_id         8633
tradable_research_flag           2
roll_window_flag                 2
low_liquidity_warning_flag       2
dtype: int64

### 6.3 Volatility and Expansion Features

In [26]:
feature_build_started = time.perf_counter()
build_result = build_mlat_feature_matrix(
    bars=bars,
    observations=eligible,
    registry=registry,
)
mlat_features = build_result.matrix
feature_build_seconds = time.perf_counter() - feature_build_started
assert mlat_features["observation_id"].tolist() == eligible["observation_id"].tolist()
mlat_features.shape

(424306, 24)

### 6.4 Volume and Activity Features

In [27]:
build_result.construction_audit, build_result.observation_audit.head()

(                        feature_name       calculation_scope  \
 0                bollinger_zscore_20  full sorted GC history   
 1             bollinger_bandwidth_20  full sorted GC history   
 2                      cutler_rsi_14  full sorted GC history   
 3              chaikin_money_flow_20  full sorted GC history   
 4              amihud_illiquidity_60  full sorted GC history   
 5            parkinson_volatility_30  full sorted GC history   
 6      rogers_satchell_volatility_30  full sorted GC history   
 7   realized_semivariance_balance_60  full sorted GC history   
 8              bipower_jump_ratio_60  full sorted GC history   
 9                variance_ratio_60_5  full sorted GC history   
 10            return_sign_entropy_60  full sorted GC history   
 11       volatility_of_volatility_60  full sorted GC history   
 
                                mapping_scope             availability  \
 0   exact decision_bar_id after construction  close of decision bar t   
 1   

### 6.5 Adaptive or Filtered-State Features

In [28]:
construction_performance = pd.DataFrame([{
    "rows_bars_gc": len(bars),
    "rows_observations": len(eligible),
    "features": len(MLAT_FEATURE_NAMES),
    "construction_seconds": feature_build_seconds,
    "rss_bytes_after": PROCESS.memory_info().rss,
    "rss_delta_bytes": PROCESS.memory_info().rss - START_RSS_BYTES,
}])
construction_performance

,rows_bars_gc,rows_observations,features,construction_seconds,rss_bytes_after,rss_delta_bytes
0,1759671,424306,12,10.258311,1084755968,827211776


### 6.6 Experimental MLAT Features

In [29]:
feature_path = PATHS.data_path("feature_matrix_mlat_gc")
feature_manifest_entry = save_parquet(
    mlat_features,
    feature_path,
    id_columns=("observation_id",),
    manifest_path=PATHS.manifest_path,
)
feature_save_checks = verify_saved_table(
    mlat_features,
    feature_path,
    id_columns=("observation_id",),
    manifest_entry=feature_manifest_entry,
)
reloaded_features = pd.read_parquet(feature_path)
reload_checks = verify_reloaded_feature_matrix(
    mlat_features,
    reloaded_features,
    registry_frame,
)
assert reload_checks["passed"].all()
feature_save_checks, reload_checks

(                     check_name  passed  \
 0                   file_exists    True   
 1                        sha256    True   
 2                     byte_size    True   
 3                     row_count    True   
 4                  column_count    True   
 5                  column_order    True   
 6                 pandas_dtypes    True   
 7                   null_counts    True   
 8           id_values_and_order    True   
 9              timestamps_exact    True   
 10           schema_fingerprint    True   
 11    deterministic_sample_hash    True   
 12             full_table_exact    True   
 13  manifest_schema_fingerprint    True   
 14         manifest_sample_hash    True   
 
                                              expected  \
 0                                                True   
 1   04cd6fc2047fa2f3dedeb454710658a6440ae970e1f418...   
 2                                            34202243   
 3                                              424306   
 4  

### 6.7 Feature Matrix Assembly

In [30]:
feature_summary = mlat_features[list(MLAT_FEATURE_NAMES)].describe().T
feature_summary["missing_fraction"] = (
    mlat_features[list(MLAT_FEATURE_NAMES)].isna().mean()
)
missingness_figure = PATHS.figure_path("feature_missingness")
figure, axis = plt.subplots(figsize=(9, 5))
feature_summary["missing_fraction"].sort_values().plot.barh(
    ax=axis, color="#35618f"
)
axis.set_title("MLAT feature missingness after eligible-observation mapping")
axis.set_xlabel("Missing fraction")
figure.tight_layout()
figure.savefig(missingness_figure, dpi=160)
plt.close(figure)
feature_summary

,count,mean,std,min,25%,50%,75%,max,missing_fraction
bollinger_zscore_20,424133.0,0.003867,1.322718,-4.335307,-1.067908,0.015053,1.078209,4.319009,0.000408
bollinger_bandwidth_20,424133.0,0.001811,0.001377,0.000140,0.000966,0.001441,0.002208,0.024496,0.000408
cutler_rsi_14,424180.0,50.147575,16.297583,0.000000,38.775509,50.000000,61.538460,100.000000,0.000297
chaikin_money_flow_20,424133.0,0.004692,0.162870,-0.748814,-0.106716,0.004456,0.115953,0.732018,0.000408
amihud_illiquidity_60,423702.0,0.888528,0.435145,0.136290,0.558913,0.805505,1.132375,4.240716,0.001424
parkinson_volatility_30,424047.0,2.654242,1.480329,0.472980,1.649812,2.274129,3.272578,21.636385,0.000610
rogers_satchell_volatility_30,424047.0,2.655442,1.501454,0.371083,1.635085,2.276885,3.289785,23.668699,0.000610
realized_semivariance_balance_60,423702.0,0.009332,0.255375,-0.970532,-0.163988,0.009915,0.179453,0.992347,0.001424
bipower_jump_ratio_60,423690.0,0.067560,0.095380,0.000000,0.000000,0.021543,0.109888,0.960133,0.001452
variance_ratio_60_5,423654.0,0.889353,0.268880,0.134498,0.696513,0.859322,1.047822,3.754612,0.001537


## 7.0 Engineering and Leakage Validation

### 7.1 Timestamp and Next-Bar Alignment

In [31]:
validation_result = validate_mlat_feature_matrix(
    mlat_features,
    registry_frame,
    eligible_observations=eligible,
    existing_feature_matrix=existing_features,
)
assert validation_result.ready
validation_result.checks

,check,passed,severity,observed,expected,details
0,registry_matrix_column_set,True,ERROR,"{'missing_metadata': [], 'missing_features': [...",exact metadata plus frozen registry feature set,
1,registry_feature_order,True,ERROR,"[observation_id, decision_timestamp_utc, decis...","[observation_id, decision_timestamp_utc, decis...",
2,observation_id_unique,True,ERROR,0,0,
3,observation_id_order,True,ERROR,True,True,
4,eligible_observation_id_alignment,True,ERROR,"{'matrix_rows': 424306, 'eligible_rows': 424306}",identical IDs in identical order,
5,eligible_decision_timestamp_utc_alignment,True,ERROR,0,0,
6,eligible_decision_timestamp_ny_alignment,True,ERROR,0,0,
7,eligible_entry_timestamp_utc_alignment,True,ERROR,0,0,
8,eligible_entry_timestamp_ny_alignment,True,ERROR,0,0,
9,eligible_trade_date_ny_alignment,True,ERROR,0,0,


### 7.2 Future-Bar Mutation Tests

In [32]:
validation_result.feature_diagnostics

,feature_name,dtype,expected_dtype,dtype_pass,null_count,missing_rate,missing_rate_pass,nonfinite_count,minimum,maximum,validation_minimum,validation_maximum,below_minimum_count,above_maximum_count,range_pass,finite_unique_count,constant
0,bollinger_zscore_20,float32,float32,True,173,0.000408,True,0,-4.335307,4.319009,NaN,NaN,0,0,True,420756,False
1,bollinger_bandwidth_20,float32,float32,True,173,0.000408,True,0,0.000140,0.024496,0.0,NaN,0,0,True,410625,False
2,cutler_rsi_14,float32,float32,True,126,0.000297,True,0,0.000000,100.000000,0.0,100.0,0,0,True,10059,False
3,chaikin_money_flow_20,float32,float32,True,173,0.000408,True,0,-0.748814,0.732018,-1.0,1.0,0,0,True,422015,False
4,amihud_illiquidity_60,float32,float32,True,604,0.001424,True,0,0.136290,4.240716,0.0,NaN,0,0,True,414414,False
5,parkinson_volatility_30,float32,float32,True,259,0.000610,True,0,0.472980,21.636385,0.0,NaN,0,0,True,418570,False
6,rogers_satchell_volatility_30,float32,float32,True,259,0.000610,True,0,0.371083,23.668699,0.0,NaN,0,0,True,414377,False
7,realized_semivariance_balance_60,float32,float32,True,604,0.001424,True,0,-0.970532,0.992347,-1.0,1.0,0,0,True,417128,False
8,bipower_jump_ratio_60,float32,float32,True,616,0.001452,True,0,0.000000,0.960133,0.0,1.0,0,0,True,237313,False
9,variance_ratio_60_5,float32,float32,True,652,0.001537,True,0,0.134498,3.754612,0.0,NaN,0,0,True,416741,False


### 7.3 Entry-Bar Mutation Tests

In [33]:
validation_result.sample_hash

,algorithm,sample_method,sample_size,row_count,column_count,hashed_column_count,schema_complete,digest
0,sha256,evenly_spaced_rows,257,424306,24,24,True,f547451f889ef5efea1176b4739814f7375c213410f045...


### 7.4 Contract and Segment Boundaries

In [34]:
print(
    "Mutation and invariance checks are executable unit tests in "
    "tests/test_mlat_feature_engineering.py."
)

Mutation and invariance checks are executable unit tests in tests/test_mlat_feature_engineering.py.


### 7.5 Missing-Bar and Session Boundaries

In [35]:
print(
    "Missing-minute, contract, segment, roll, tradability, liquidity, "
    "New York date, and DST cases are covered by the MLAT test suite."
)

Missing-minute, contract, segment, roll, tradability, liquidity, New York date, and DST cases are covered by the MLAT test suite.


### 7.6 Numerical and Missingness Checks

In [36]:
reload_checks, validation_result.sample_hash

(                               check  passed severity  \
 0                       reload_shape    True    ERROR   
 1                reload_column_order    True    ERROR   
 2                      reload_dtypes    True    ERROR   
 3              reload_observation_id    True    ERROR   
 4      reload_decision_timestamp_utc    True    ERROR   
 5         reload_entry_timestamp_utc    True    ERROR   
 6                 reload_null_counts    True    ERROR   
 7             reload_metadata_values    True    ERROR   
 8           reload_nonnumeric_values    True    ERROR   
 9   reload_deterministic_sample_hash    True    ERROR   
 10             reload_numeric_values    True    ERROR   
 
                                              observed  \
 0                                        (424306, 24)   
 1   [observation_id, decision_timestamp_utc, decis...   
 2                                                  []   
 3                                                   0   
 4          

### 7.7 Save and Reload Validation

In [37]:
ENGINEERING_GATE = bool(
    REGISTRY_VALID
    and validation_result.ready
    and reload_checks["passed"].all()
)
assert ENGINEERING_GATE
ENGINEERING_GATE

True

## 8.0 MLAT Univariate Feature Evaluation

### 8.1 Development Screen

In [38]:
EVALUATION_CONFIG = MlatEvaluationConfig(
    horizons=HORIZONS,
    seed=SEED,
    bootstrap_iterations=2_000,
)
evaluation_result = build_mlat_evaluation_frame(
    mlat_features,
    labels,
    registry_frame,
    existing_features,
    frozen_features=frozen_expansion,
    config=EVALUATION_CONFIG,
)
assert evaluation_result.checks["passed"].all()
evaluation_frame = evaluation_result.frame
assert set(evaluation_frame["research_partition"]) == set(PARTITIONS)
assert evaluation_frame["product"].eq("GC").all()
evaluation_result.checks, evaluation_frame.shape

(                                check  passed  \
 0                             gc_only    True   
 1          feature_label_id_alignment    True   
 2  feature_label_provenance_alignment    True   
 3                      final_excluded    True   
 4          unknown_partition_excluded    True   
 5      new_york_date_evidence_present    True   
 
                                             observed  \
 0  {'feature_products': ['GC'], 'label_products':...   
 1               {'feature_only': 0, 'label_only': 0}   
 2  {'decision_timestamp_utc': 0, 'decision_timest...   
 3                                                  0   
 4                                                  0   
 5                                                  0   
 
                                             expected  
 0  {'feature_products': ['GC'], 'label_products':...  
 1               {'feature_only': 0, 'label_only': 0}  
 2  {'decision_timestamp_utc': 0, 'decision_timest...  
 3                     

### 8.2 Validation Confirmation

In [39]:
cell_results = evaluate_mlat_feature_cells(
    evaluation_result,
    registry_frame,
    config=EVALUATION_CONFIG,
    run_horizon_thinning=True,
)
cell_results.daily_ic.head()

,trade_date_ny,observations,daily_spearman_ic,eligible_date,feature_name,target_family,horizon_minutes,entry_session,research_partition
0,2021-05-24,180,-0.056458,True,bollinger_zscore_20,direction,5,London,Development
1,2021-05-25,180,-0.072937,True,bollinger_zscore_20,direction,5,London,Development
2,2021-05-26,180,0.084914,True,bollinger_zscore_20,direction,5,London,Development
3,2021-05-31,180,-0.157575,True,bollinger_zscore_20,direction,5,London,Development
4,2021-06-01,180,0.138678,True,bollinger_zscore_20,direction,5,London,Development


### 8.3 Directional Outcomes

In [40]:
cell_results.cell_results[[
    "feature_name", "target_family", "horizon_minutes", "entry_session",
    "research_partition", "mean_daily_ic", "bootstrap_ci_lower",
    "bootstrap_ci_upper", "p_value", "q_value",
]].head()

,feature_name,target_family,horizon_minutes,entry_session,research_partition,mean_daily_ic,bootstrap_ci_lower,bootstrap_ci_upper,p_value,q_value
0,bollinger_zscore_20,direction,5,London,Development,-0.078591,-0.090023,-0.067097,0.001,0.002999
1,bollinger_zscore_20,direction,5,London,Validation,-0.068101,-0.086465,-0.050566,0.001,NaN
2,bollinger_zscore_20,direction,15,London,Development,-0.110273,-0.126069,-0.094105,0.001,0.002999
3,bollinger_zscore_20,direction,15,London,Validation,-0.118004,-0.145553,-0.091532,0.001,NaN
4,bollinger_zscore_20,direction,30,London,Development,-0.150628,-0.167858,-0.133633,0.001,0.002999


### 8.4 Expansion Outcomes

In [41]:
cell_results.quintile_results.head()

,feature_name,target_family,horizon_minutes,entry_session,research_partition,edges_fit_partition,observations,q1_mean,q2_mean,q3_mean,q4_mean,q5_mean,absolute_monotonicity,top_bottom_effect_size,top_bottom_tick_spread
0,bollinger_zscore_20,direction,5,London,Development,Development,113872,0.018238,0.019372,0.025614,-0.004511,-0.032230,0.6,-0.029939,-0.255998
1,bollinger_zscore_20,direction,5,London,Validation,Development,43615,0.063624,0.027573,-0.003886,0.007364,0.044034,0.3,-0.011890,-0.245576
2,bollinger_zscore_20,direction,15,London,Development,Development,113872,0.008521,0.010339,0.037448,-0.011011,-0.041055,0.6,-0.017645,-0.275500
3,bollinger_zscore_20,direction,15,London,Validation,Development,43615,0.135353,0.119289,0.110428,-0.006699,0.061333,0.9,-0.026049,-0.656824
4,bollinger_zscore_20,direction,30,London,Development,Development,113872,0.007272,0.029020,0.094262,-0.008315,-0.092243,0.6,-0.025141,-0.626234


### 8.5 Risk-State Outcomes

In [42]:
cell_results.cell_results.head()

,feature_name,target_family,horizon_minutes,entry_session,research_partition,finite_observations,eligible_dates,mean_daily_ic,bootstrap_ci_lower,bootstrap_ci_upper,...,q2_mean,q3_mean,q4_mean,q5_mean,absolute_monotonicity,top_bottom_effect_size,top_bottom_tick_spread,thinned_mean_daily_ic,thinning_sign_agreement,q_value
0,bollinger_zscore_20,direction,5,London,Development,113870,636,-0.078591,-0.090023,-0.067097,...,0.019372,0.025614,-0.004511,-0.032230,0.6,-0.029939,-0.255998,-0.071753,True,0.002999
1,bollinger_zscore_20,direction,5,London,Validation,43612,244,-0.068101,-0.086465,-0.050566,...,0.027573,-0.003886,0.007364,0.044034,0.3,-0.011890,-0.245576,-0.060582,True,NaN
2,bollinger_zscore_20,direction,15,London,Development,113870,636,-0.110273,-0.126069,-0.094105,...,0.010339,0.037448,-0.011011,-0.041055,0.6,-0.017645,-0.275500,-0.096547,True,0.002999
3,bollinger_zscore_20,direction,15,London,Validation,43612,244,-0.118004,-0.145553,-0.091532,...,0.119289,0.110428,-0.006699,0.061333,0.9,-0.026049,-0.656824,-0.098641,True,NaN
4,bollinger_zscore_20,direction,30,London,Development,113870,636,-0.150628,-0.167858,-0.133633,...,0.029020,0.094262,-0.008315,-0.092243,0.6,-0.025141,-0.626234,NaN,False,0.002999


### 8.6 Session Stability

In [43]:
cell_results.year_stability.head()

,feature_name,target_family,horizon_minutes,entry_session,research_partition,year,eligible_dates,mean_daily_ic,full_period_sign_agreement
0,bollinger_zscore_20,direction,5,London,Development,2021,151,-0.066582,True
1,bollinger_zscore_20,direction,5,London,Development,2022,244,-0.082864,True
2,bollinger_zscore_20,direction,5,London,Development,2023,241,-0.081788,True
3,bollinger_zscore_20,direction,5,London,Validation,2024,244,-0.068101,True
4,bollinger_zscore_20,direction,15,London,Development,2021,151,-0.104048,True


### 8.7 Year Stability

In [44]:
cell_results.thinning_results.head()

,feature_name,target_family,horizon_minutes,entry_session,research_partition,full_mean_daily_ic,thinned_mean_daily_ic,thinned_dates,thinned_observations,sign_agreement
0,bollinger_zscore_20,direction,5,London,Development,-0.078591,-0.071753,632,22750,True
1,bollinger_zscore_20,direction,5,London,Validation,-0.068101,-0.060582,242,8712,True
2,bollinger_zscore_20,direction,15,London,Development,-0.110273,-0.096547,632,7583,True
3,bollinger_zscore_20,direction,15,London,Validation,-0.118004,-0.098641,242,2904,True
4,bollinger_zscore_20,direction,30,London,Development,-0.150628,NaN,0,0,False


### 8.8 Multiple-Testing Control

In [45]:
evaluation_pairs = cell_results.cell_results.pivot_table(
    index=[
        "feature_name", "target_family", "horizon_minutes", "entry_session"
    ],
    columns="research_partition",
    values="mean_daily_ic",
).dropna()
ic_comparison_figure = PATHS.figure_path("development_validation_ic")
figure, axis = plt.subplots(figsize=(7, 7))
for family, group in evaluation_pairs.reset_index().groupby("target_family"):
    axis.scatter(
        group["Development"],
        group["Validation"],
        s=18,
        alpha=0.65,
        label=family,
    )
limits = np.nanmax(np.abs(evaluation_pairs[["Development", "Validation"]].to_numpy()))
limits = max(float(limits), 0.01)
axis.plot([-limits, limits], [-limits, limits], "--", color="0.45", linewidth=1)
axis.axhline(0, color="0.75", linewidth=0.8)
axis.axvline(0, color="0.75", linewidth=0.8)
axis.set(
    xlabel="Development mean daily Spearman IC",
    ylabel="Validation mean daily Spearman IC",
    title="Development versus Validation feature evidence",
    xlim=(-limits, limits),
    ylim=(-limits, limits),
)
axis.legend(frameon=False)
figure.tight_layout()
figure.savefig(ic_comparison_figure, dpi=160)
plt.close(figure)

cell_results.cell_results.groupby(
    ["target_family", "research_partition", "entry_session"]
)["mean_daily_ic"].agg(["count", "mean", "median"])

count      mean    median
target_family research_partition entry_session                           
direction     Development        London            72 -0.053199 -0.005788
                                 New York          72 -0.042031 -0.006909
              Validation         London            72 -0.053758 -0.012321
                                 New York          72 -0.043469 -0.014594
expansion     Development        London            72 -0.081749 -0.012479
                                 New York          72 -0.069824 -0.015225
              Validation         London            72 -0.090752 -0.042938
                                 New York          72 -0.072453 -0.024691
path_risk     Development        London            72 -0.068254 -0.011647
                                 New York          72 -0.060982 -0.010484
              Validation         London            72 -0.075406 -0.037929
                                 New York          72 -0.063671 -0.026658
volatility    Development        London            72  0.009230  0.003649
                                 New York          72  0.062993  0.002087
              Validation         London            72 -0.003416 -0.011434
                                 New York          72  0.054049  0.000559

## 9.0 Redundancy and Incremental Information

### 9.1 Correlation Structure

In [46]:
overlap_result = audit_feature_overlap(
    mlat_features,
    existing_features,
    candidate_features=MLAT_FEATURE_NAMES,
    existing_features=existing_names,
    partition_column="research_partition",
    near_duplicate_threshold=EVALUATION_CONFIG.overlap_threshold,
)
existing_overlap_summary = (
    overlap_result.loc[
        overlap_result["reference_scope"].eq("existing")
        & overlap_result["audit_type"].eq("overlap")
    ]
    .sort_values("absolute_correlation", ascending=False)
    .groupby("candidate_feature", as_index=False)
    .first()
)
existing_overlap_summary

,candidate_feature,reference_feature,reference_scope,audit_type,development_observations,minimum_observations,sufficient_overlap,spearman_correlation,absolute_correlation,is_exact_duplicate,is_near_duplicate,is_constant,threshold
0,amihud_illiquidity_60,time_of_day_sin,existing,overlap,305853,30,True,0.634153,0.634153,False,False,False,0.995
1,bipower_jump_ratio_60,return_sign_change_rate_30,existing,overlap,305846,30,True,0.108195,0.108195,False,False,False,0.995
2,bollinger_bandwidth_20,realized_volatility_15,existing,overlap,306114,30,True,0.808396,0.808396,False,False,False,0.995
3,bollinger_zscore_20,distance_from_rolling_vwap_20_atr,existing,overlap,306114,30,True,0.963953,0.963953,False,False,False,0.995
4,chaikin_money_flow_20,distance_from_rolling_vwap_20_atr,existing,overlap,306114,30,True,0.535819,0.535819,False,False,False,0.995
5,cutler_rsi_14,return_15m_atr,existing,overlap,306114,30,True,0.948554,0.948554,False,False,False,0.995
6,parkinson_volatility_30,atr_20,existing,overlap,306062,30,True,0.972692,0.972692,False,False,False,0.995
7,realized_semivariance_balance_60,return_60m_atr,existing,overlap,305853,30,True,0.870690,0.870690,False,False,False,0.995
8,return_sign_entropy_60,atr_60,existing,overlap,305853,30,True,-0.715162,0.715162,False,False,False,0.995
9,rogers_satchell_volatility_30,atr_20,existing,overlap,306062,30,True,0.961703,0.961703,False,False,False,0.995


### 9.2 Feature Clustering

In [47]:
development_mask = mlat_features["research_partition"].eq("Development")
candidate_correlation = mlat_features.loc[
    development_mask, list(MLAT_FEATURE_NAMES)
].corr(method="spearman")
overlap_figure = PATHS.figure_path("candidate_spearman_correlation")
figure, axis = plt.subplots(figsize=(10, 9))
image = axis.imshow(
    candidate_correlation.to_numpy(),
    vmin=-1,
    vmax=1,
    cmap="coolwarm",
    aspect="auto",
)
axis.set_xticks(range(len(MLAT_FEATURE_NAMES)))
axis.set_yticks(range(len(MLAT_FEATURE_NAMES)))
axis.set_xticklabels(MLAT_FEATURE_NAMES, rotation=90, fontsize=7)
axis.set_yticklabels(MLAT_FEATURE_NAMES, fontsize=7)
axis.set_title("Development Spearman correlation: MLAT v1 candidates")
figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
figure.tight_layout()
figure.savefig(overlap_figure, dpi=160)
plt.close(figure)
candidate_correlation

,bollinger_zscore_20,bollinger_bandwidth_20,cutler_rsi_14,chaikin_money_flow_20,amihud_illiquidity_60,parkinson_volatility_30,rogers_satchell_volatility_30,realized_semivariance_balance_60,bipower_jump_ratio_60,variance_ratio_60_5,return_sign_entropy_60,volatility_of_volatility_60
bollinger_zscore_20,1.000000,-0.002878,0.879968,0.492414,-0.002214,-0.006544,-0.008430,0.349538,0.009829,0.006307,0.007913,-0.005085
bollinger_bandwidth_20,-0.002878,1.000000,-0.001117,-0.005728,-0.371130,0.766833,0.741117,-0.028763,0.038661,0.245365,-0.497382,0.211910
cutler_rsi_14,0.879968,-0.001117,1.000000,0.507187,-0.002579,-0.007801,-0.009779,0.364930,0.011261,0.007453,0.009869,-0.004575
chaikin_money_flow_20,0.492414,-0.005728,0.507187,1.000000,-0.007850,-0.010724,-0.011600,0.219205,0.007664,0.005090,0.011094,-0.001216
amihud_illiquidity_60,-0.002214,-0.371130,-0.002579,-0.007850,1.000000,-0.501876,-0.516619,0.014091,-0.030923,-0.148407,0.346662,-0.195368
parkinson_volatility_30,-0.006544,0.766833,-0.007801,-0.010724,-0.501876,1.000000,0.987436,-0.039557,0.018310,0.100170,-0.652555,0.278945
rogers_satchell_volatility_30,-0.008430,0.741117,-0.009779,-0.011600,-0.516619,0.987436,1.000000,-0.041181,0.012772,0.108130,-0.640763,0.272497
realized_semivariance_balance_60,0.349538,-0.028763,0.364930,0.219205,0.014091,-0.039557,-0.041181,1.000000,0.033321,0.013043,0.025991,-0.010692
bipower_jump_ratio_60,0.009829,0.038661,0.011261,0.007664,-0.030923,0.018310,0.012772,0.033321,1.000000,-0.014853,0.232529,0.117389
variance_ratio_60_5,0.006307,0.245365,0.007453,0.005090,-0.148407,0.100170,0.108130,0.013043,-0.014853,1.000000,-0.099001,0.021910


### 9.3 Incremental Value Beyond Existing Features

In [48]:
incremental_result = evaluate_incremental_information(
    evaluation_result,
    registry_frame,
    atr_anchor="atr_20",
    frozen_features=frozen_expansion,
    config=EVALUATION_CONFIG,
    horizons=(60, 180),
)
incremental_result.summary.loc[
    incremental_result.summary["control_set"].eq("frozen_15")
]

,feature_name,target_family,horizon_minutes,entry_session,control_set,eligible_dates_development,eligible_dates_validation,finite_observations_development,finite_observations_validation,mean_daily_partial_ic_development,mean_daily_partial_ic_validation,development_threshold,development_sample_pass,validation_sample_pass,development_magnitude_pass,validation_sign_agreement,validation_magnitude_retention,validation_retention_pass,incremental_pass
1,amihud_illiquidity_60,direction,60,London,frozen_15,635,244,113793,43612,0.017429,-0.029854,0.01,True,True,True,False,1.712847,True,False
3,amihud_illiquidity_60,direction,60,New York,frozen_15,638,246,188760,72895,0.009425,0.040806,0.01,True,True,False,True,4.329592,True,False
5,amihud_illiquidity_60,direction,180,London,frozen_15,635,244,113793,43612,0.008853,-0.006160,0.01,True,True,False,False,0.695842,True,False
7,amihud_illiquidity_60,direction,180,New York,frozen_15,638,246,188760,72895,0.026704,0.073915,0.01,True,True,True,True,2.767957,True,True
9,amihud_illiquidity_60,expansion,60,London,frozen_15,635,244,113793,43612,0.087460,0.102946,0.01,True,True,True,True,1.177062,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,volatility_of_volatility_60,path_risk,180,New York,frozen_15,638,246,188704,72839,-0.011718,-0.000291,0.01,True,True,True,True,0.024815,False,False
377,volatility_of_volatility_60,volatility,60,London,frozen_15,635,244,113779,43612,0.026181,0.022285,0.01,True,True,True,True,0.851170,True,True
379,volatility_of_volatility_60,volatility,60,New York,frozen_15,638,246,188704,72839,-0.038798,-0.029458,0.01,True,True,True,True,0.759275,True,True
381,volatility_of_volatility_60,volatility,180,London,frozen_15,635,244,113779,43612,0.035485,0.020868,0.01,True,True,True,True,0.588081,True,True


### 9.4 Incremental Value Beyond ATR and Expansion Anchors

In [49]:
incremental_result.summary.loc[
    incremental_result.summary["control_set"].eq("atr_20")
]

,feature_name,target_family,horizon_minutes,entry_session,control_set,eligible_dates_development,eligible_dates_validation,finite_observations_development,finite_observations_validation,mean_daily_partial_ic_development,mean_daily_partial_ic_validation,development_threshold,development_sample_pass,validation_sample_pass,development_magnitude_pass,validation_sign_agreement,validation_magnitude_retention,validation_retention_pass,incremental_pass
0,amihud_illiquidity_60,direction,60,London,atr_20,635,244,113793,43612,0.009451,0.029968,0.02,True,True,False,True,3.170780,True,False
2,amihud_illiquidity_60,direction,60,New York,atr_20,638,246,188760,72895,0.017390,0.045983,0.02,True,True,False,True,2.644226,True,False
4,amihud_illiquidity_60,direction,180,London,atr_20,635,244,113793,43612,0.007323,0.017185,0.02,True,True,False,True,2.346772,True,False
6,amihud_illiquidity_60,direction,180,New York,atr_20,638,246,188760,72895,0.020521,0.067345,0.02,True,True,True,True,3.281762,True,True
8,amihud_illiquidity_60,expansion,60,London,atr_20,635,244,113793,43612,0.098748,0.113479,0.02,True,True,True,True,1.149184,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374,volatility_of_volatility_60,path_risk,180,New York,atr_20,638,246,188704,72839,0.096741,0.092212,0.02,True,True,True,True,0.953192,True,True
376,volatility_of_volatility_60,volatility,60,London,atr_20,635,244,113779,43612,0.035920,-0.006420,0.02,True,True,True,False,0.178736,False,False
378,volatility_of_volatility_60,volatility,60,New York,atr_20,638,246,188704,72839,0.052904,0.065999,0.02,True,True,True,True,1.247508,True,True
380,volatility_of_volatility_60,volatility,180,London,atr_20,635,244,113779,43612,0.029602,-0.012286,0.02,True,True,True,False,0.415048,True,False


### 9.5 Frozen MLAT Shortlist

In [50]:
preliminary_verdicts = assign_mlat_verdicts(
    cell_results,
    incremental_result,
    overlap_result,
    validation_result,
    registry=registry_frame,
    config=EVALUATION_CONFIG,
)
ADVANCEMENT_GATE_FEASIBLE = bool(
    preliminary_verdicts["authorization_gate_open"].astype(bool).all()
)
authorization_reasons = (
    preliminary_verdicts["authorization_gate_reason"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)
assert len(authorization_reasons) == 1
ADVANCEMENT_GATE_REASON = authorization_reasons[0]
{
    "advancement_gate_feasible": ADVANCEMENT_GATE_FEASIBLE,
    "reason": ADVANCEMENT_GATE_REASON,
    "verdict_counts": preliminary_verdicts["verdict"].value_counts().to_dict(),
}

{'advancement_gate_feasible': False,
 'reason': 'frozen v1 horizon-thinning gate is structurally non-evaluable: 0/384 required 60/180-minute feature-family-session-partition cells have finite thinned daily-IC evidence under the >=10 observations-per-New-York-date rule, including 0 Development cells; advancement is not authorized',
 'verdict_counts': {'RESEARCH_ONLY': 12}}

## 10.0 MLAT Volatility Model Audit

### 10.1 Existing GARCH Implementation Audit

In [51]:
print(
    (DOCS / "mlat_current_state_audit.md")
    .read_text(encoding="utf-8")
    .split("## GARCH", 1)[-1][:2_000]
)

# MLAT Current-State Audit

Audit date: 2026-07-23. Classifications reflect the current working tree and
the physical saved artifacts, not assumptions from notebook prose alone.

| Statement | Classification | Evidence and qualification |
|---|---|---|
| The trusted data are Databento CME futures one-minute OHLCV bars. | VERIFIED WITH QUALIFICATION | `data/processed/research_bars_gc_mgc_1m.parquet` contains 3,487,656 one-minute active-contract rows with OHLCV, Databento identifiers, CME/Globex lineage, and project-added controls. It is a processed active-contract research table, not raw trade, quote, MBO, or order-book data. |
| GC is the discovery instrument. | VERIFIED | The eligible-observation, label, feature, baseline, and frozen-feature artifacts are GC-only. The statistical context report freezes GC as Phase 1 discovery. |
| MGC is reserved for later transfer validation and execution mapping. | VERIFIED | The statistical context report and saved artifact names enforce this role.

### 10.2 Model Specification Diagnostics

In [52]:
garch_validation_end = pd.Timestamp(
    "2024-12-31 23:59:59", tz="America/New_York"
).tz_convert("UTC")
garch_bars = bars.loc[
    pd.to_datetime(bars["ts_event_utc"], utc=True).le(garch_validation_end)
].copy()
with warnings.catch_warnings(record=True) as garch_warnings:
    warnings.simplefilter("always")
    garch_audit = audit_segmented_garch(
        garch_bars,
        fit_end="2022-12-31 23:59:59-05:00",
        audit_start="2023-01-01 00:00:00-05:00",
        audit_end="2023-12-31 23:59:59-05:00",
        validation_start="2024-01-01 00:00:00-05:00",
        validation_end="2024-12-31 23:59:59-05:00",
    )
assert pd.to_datetime(
    garch_audit.forecasts["ts_event_utc"], utc=True
).max() <= garch_validation_end
garch_audit.fit.diagnostics, garch_audit.residual_diagnostics

(    category                     check_name  critical  passed  numeric_value  \
 0  garch_fit                  gc_only_input      True    True   1.000000e+00   
 1  garch_fit            fit_scope_ends_2022      True    True   1.672438e+18   
 2  garch_fit              optimizer_success      True    True   0.000000e+00   
 3  garch_fit               objective_finite      True    True   2.112386e+00   
 4  garch_fit                 omega_positive      True    True   3.474831e-10   
 5  garch_fit              alpha_nonnegative      True    True   7.600240e-02   
 6  garch_fit               beta_nonnegative      True    True   9.139219e-01   
 7  garch_fit          persistence_below_one      True    True   9.899243e-01   
 8  garch_fit  persistence_below_audit_limit      True    True   9.899243e-01   
 
                                      threshold  \
 0  products == {'GC'} after explicit filtering   
 1       timestamp <= 2023-01-01T04:59:59+00:00   
 2                        scipy suc

### 10.3 Point-in-Time Forecast Construction

In [53]:
garch_audit.forecasts.head()

,source_row_id,ts_event_utc,product,open,high,low,close,source_continuity_run_id,continuity_run_id,continuity_run_position,...,garch_sigma_forecast_60m,standardized_residual,realized_volatility_60m,target_60m_end_timestamp_utc,rv_anchor_sigma_60m,atr_anchor_sigma_60m,audit_period,garch_sigma_calibrated_60m,rv_anchor_sigma_calibrated_60m,atr_anchor_sigma_calibrated_60m
0,0,2021-05-24 00:00:00+00:00,GC,1884.2,1884.5,1883.9,1884.0,0,0,0,...,0.001438,NaN,0.001266,2021-05-24 01:00:00+00:00,NaN,NaN,fit,0.001391,NaN,NaN
1,1,2021-05-24 00:01:00+00:00,GC,1884.1,1884.1,1882.9,1883.3,0,0,1,...,0.001557,-2.001912,0.001420,2021-05-24 01:01:00+00:00,NaN,NaN,fit,0.001506,NaN,NaN
2,2,2021-05-24 00:02:00+00:00,GC,1883.5,1883.5,1883.1,1883.4,0,0,2,...,0.001512,0.257220,0.001467,2021-05-24 01:02:00+00:00,NaN,NaN,fit,0.001462,NaN,NaN
3,3,2021-05-24 00:03:00+00:00,GC,1883.3,1883.4,1883.2,1883.4,0,0,3,...,0.001466,-0.000759,0.001543,2021-05-24 01:03:00+00:00,NaN,NaN,fit,0.001417,NaN,NaN
4,4,2021-05-24 00:04:00+00:00,GC,1883.4,1883.8,1883.4,1883.8,0,0,4,...,0.001476,1.114577,0.001528,2021-05-24 01:04:00+00:00,NaN,NaN,fit,0.001427,NaN,NaN


### 10.4 Validation and Calibration

In [54]:
validation_benchmarks = garch_audit.benchmark_metrics.loc[
    garch_audit.benchmark_metrics["period"].eq("validation")
].copy()
garch_figure = PATHS.figure_path("garch_validation_benchmarks")
figure, axis = plt.subplots(figsize=(8, 5))
validation_benchmarks.plot.bar(
    x="method",
    y="rmse_sigma",
    legend=False,
    color="#8b4f6c",
    ax=axis,
)
axis.set_title("2024 validation volatility-forecast RMSE")
axis.set_ylabel("RMSE (volatility scale)")
axis.set_xlabel("")
figure.tight_layout()
figure.savefig(garch_figure, dpi=160)
plt.close(figure)
garch_audit.calibration, garch_audit.benchmark_metrics

(                       method       raw_forecast_column  \
 0                       garch  garch_sigma_forecast_60m   
 1  realized_volatility_anchor       rv_anchor_sigma_60m   
 2                  atr_anchor      atr_anchor_sigma_60m   
 
         calibrated_forecast_column                        calibration_method  \
 0       garch_sigma_calibrated_60m  nonnegative_zero_intercept_least_squares   
 1   rv_anchor_sigma_calibrated_60m  nonnegative_zero_intercept_least_squares   
 2  atr_anchor_sigma_calibrated_60m  nonnegative_zero_intercept_least_squares   
 
       calibration_start_utc       calibration_end_utc  n_fit_observations  \
 0 2023-01-01 05:00:00+00:00 2024-01-01 04:59:59+00:00              295599   
 1 2023-01-01 05:00:00+00:00 2024-01-01 04:59:59+00:00              275427   
 2 2023-01-01 05:00:00+00:00 2024-01-01 04:59:59+00:00              288874   
 
    multiplier  intercept  success  \
 0    0.966844        0.0     True   
 1    0.903134        0.0     True   
 2  

### 10.5 GARCH Feature Verdict

In [55]:
critical_garch_checks = garch_audit.audit_checks["critical"].astype(bool)
GARCH_AUDIT_PASS = bool(
    garch_audit.audit_checks.loc[
        critical_garch_checks, "passed"
    ].astype(bool).all()
)
GARCH_VERDICT = (
    "RESEARCH_ONLY" if GARCH_AUDIT_PASS else "REJECTED_IMPLEMENTATION"
)
{
    "verdict": GARCH_VERDICT,
    "audit_checks": garch_audit.audit_checks.to_dict("records"),
    "warnings": [str(item.message) for item in garch_warnings],
}

{'verdict': 'REJECTED_IMPLEMENTATION',
 'audit_checks': [{'category': 'garch_audit',
   'check_name': 'fit_critical_checks',
   'critical': True,
   'passed': True,
   'details': 'all optimizer and parameter gates',
   'advancement_authorized': False},
  {'category': 'garch_audit',
   'check_name': 'fit_residual_diagnostics',
   'critical': True,
   'passed': False,
   'details': 'segment-aware Ljung-Box on residuals/squares and ARCH LM',
   'advancement_authorized': False},
  {'category': 'garch_audit',
   'check_name': 'development_audit_residual_diagnostics',
   'critical': True,
   'passed': False,
   'details': 'segment-aware Ljung-Box on residuals/squares and ARCH LM',
   'advancement_authorized': False},
  {'category': 'garch_audit',
   'check_name': 'validation_residual_diagnostics',
   'critical': True,
   'passed': False,
   'details': 'segment-aware Ljung-Box on residuals/squares and ARCH LM',
   'advancement_authorized': False},
  {'category': 'garch_audit',
   'check_name'

## 11.0 Multivariate Research Authorization Gate

### 11.1 Evidence Required to Proceed

In [56]:
advancing = preliminary_verdicts["verdict"].isin({
    "ADVANCE_DIRECTIONAL", "ADVANCE_EXPANSION", "ADVANCE_RISK_STATE"
})
n_advancing = int(advancing.sum())
MULTIVARIATE_AUTHORIZED = n_advancing >= 3
{"advancing_nonredundant_features": n_advancing}

{'advancing_nonredundant_features': 0}

### 11.2 Linear Benchmark

In [57]:
print(
    "Authorized for a later experiment."
    if MULTIVARIATE_AUTHORIZED
    else "Not run: the pre-registered authorization gate was not met."
)

Not run: the pre-registered authorization gate was not met.


### 11.3 Regularized Benchmark

In [58]:
print(
    "Deferred to a separately frozen experiment; no parameter search is run here."
)

Deferred to a separately frozen experiment; no parameter search is run here.


### 11.4 Nonlinear Model Authorization Decision

In [59]:
assert not MULTIVARIATE_AUTHORIZED or n_advancing >= 3
{
    "multivariate_authorized": MULTIVARIATE_AUTHORIZED,
    "nonlinear_model_authorized": False,
}

{'multivariate_authorized': False, 'nonlinear_model_authorized': False}

## 12.0 MLAT Feature Verdicts

### 12.1 Directional Verdicts

In [60]:
preliminary_verdicts.loc[
    preliminary_verdicts["target_family"].eq("direction")
    if "target_family" in preliminary_verdicts
    else preliminary_verdicts["verdict"].eq("ADVANCE_DIRECTIONAL")
]

,feature_name,hypothesis_id,book_chapter,source_pdf_page,validation_passed,overlap_veto,overlap_reference,any_development_screen_pass,any_validation_confirmation_pass,any_screened_validation_confirmation_pass,...,year_stability,redundancy_result,incremental_information_result,economic_interpretation,authorization_gate_open,authorization_gate_reason,pre_authorization_verdict,verdict,final_decision,verdict_reason


### 12.2 Expansion Verdicts

In [61]:
preliminary_verdicts.loc[
    preliminary_verdicts["verdict"].eq("ADVANCE_EXPANSION")
]

,feature_name,hypothesis_id,book_chapter,source_pdf_page,validation_passed,overlap_veto,overlap_reference,any_development_screen_pass,any_validation_confirmation_pass,any_screened_validation_confirmation_pass,...,year_stability,redundancy_result,incremental_information_result,economic_interpretation,authorization_gate_open,authorization_gate_reason,pre_authorization_verdict,verdict,final_decision,verdict_reason


### 12.3 Risk-State Verdicts

In [62]:
preliminary_verdicts.loc[
    preliminary_verdicts["verdict"].eq("ADVANCE_RISK_STATE")
]

,feature_name,hypothesis_id,book_chapter,source_pdf_page,validation_passed,overlap_veto,overlap_reference,any_development_screen_pass,any_validation_confirmation_pass,any_screened_validation_confirmation_pass,...,year_stability,redundancy_result,incremental_information_result,economic_interpretation,authorization_gate_open,authorization_gate_reason,pre_authorization_verdict,verdict,final_decision,verdict_reason


### 12.4 Rejected Features

In [63]:
preliminary_verdicts.loc[
    ~preliminary_verdicts["verdict"].str.startswith("ADVANCE_")
]

,feature_name,hypothesis_id,book_chapter,source_pdf_page,validation_passed,overlap_veto,overlap_reference,any_development_screen_pass,any_validation_confirmation_pass,any_screened_validation_confirmation_pass,...,year_stability,redundancy_result,incremental_information_result,economic_interpretation,authorization_gate_open,authorization_gate_reason,pre_authorization_verdict,verdict,final_decision,verdict_reason
0,bollinger_zscore_20,MLAT-H001,4 / Appendix,131-133; 740-742,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,PASS: no exact or >=0.995 existing-feature ove...,15/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,ECONOMICALLY_TRIVIAL,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...
1,bollinger_bandwidth_20,MLAT-H002,Appendix,740-742,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,PASS: no exact or >=0.995 existing-feature ove...,20/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...
2,cutler_rsi_14,MLAT-H003,4,132,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,PASS: no exact or >=0.995 existing-feature ove...,17/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,ECONOMICALLY_TRIVIAL,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...
3,chaikin_money_flow_20,MLAT-H004,Appendix,752-753,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,PASS: no exact or >=0.995 existing-feature ove...,11/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,ECONOMICALLY_TRIVIAL,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...
4,amihud_illiquidity_60,MLAT-H005,20 / Appendix,656; 752,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,PASS: no exact or >=0.995 existing-feature ove...,19/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...
5,parkinson_volatility_30,MLAT-H006,9,297-301,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,PASS: no exact or >=0.995 existing-feature ove...,24/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...
6,rogers_satchell_volatility_30,MLAT-H007,9,297-301,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,PASS: no exact or >=0.995 existing-feature ove...,22/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...
7,realized_semivariance_balance_60,MLAT-H008,5 / 9,169-178; 297-301,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,PASS: no exact or >=0.995 existing-feature ove...,15/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...
8,bipower_jump_ratio_60,MLAT-H009,9,297-301,True,False,<NA>,True,True,True,...,MIXED: at least one evaluated cell has stable ...,

### 12.5 Deferred Features

In [64]:
hypothesis_catalog.loc[
    hypothesis_catalog["decision"].astype(str).str.contains(
        "DEFER|LATER|DIFFERENT_DATA", regex=True
    )
]

,hypothesis_id,feature_name,source,source_type,target,overlap,leakage,cost,decision
12,MLAT-H013,kalman_innovation_state,"Ch. 4, PDF 133-136",MLAT-ADAPTED,direction / risk state,EMA/trend states,HIGH,HIGH,DEFER
17,MLAT-H018,kama_gap,"Ch. Appendix, PDF 738-740",MLAT-DIRECT,direction / risk state,efficiency ratios and slopes,MEDIUM,HIGH,DEFER
22,MLAT-H023,cross_sectional_risk_factor,"Ch. 4 / 13, PDF 115-130; 429-460",MLAT-DIRECT,portfolio,none,HIGH,HIGH,DEFER_DIFFERENT_DATA
23,MLAT-H024,news_sentiment,"Ch. 14-16, PDF 461-532",MLAT-DIRECT,direction / risk,none,HIGH,HIGH,DEFER_DIFFERENT_DATA
24,MLAT-H025,pca_regime_state,"Ch. 13, PDF 429-460",MLAT-ADAPTED,risk state,existing clustering/frozen set,MEDIUM,HIGH,LATER_PHASE
25,MLAT-H026,tree_or_boosted_feature_importance,"Ch. 11-12, PDF 350-428",MLAT-DIRECT,model validation,none,HIGH,HIGH,LATER_PHASE
26,MLAT-H027,deep_sequence_model,"Ch. 17-21, PDF 533-690",MLAT-DIRECT,later modelling,none,VERY HIGH,VERY HIGH,LATER_PHASE
28,MLAT-H029,mgc_transfer_check,"Ch. 1-2, PDF 42-94",PROJECT-ORIGINAL-EXTENSION,transfer validation,n/a,MEDIUM,HIGH,LATER_PHASE


## 13.0 MLAT Conclusions and Handoff

### 13.1 Main Findings

In [65]:
final_verdicts = preliminary_verdicts.copy()
final_verdicts["garch_audit_verdict"] = GARCH_VERDICT
final_verdicts

,feature_name,hypothesis_id,book_chapter,source_pdf_page,validation_passed,overlap_veto,overlap_reference,any_development_screen_pass,any_validation_confirmation_pass,any_screened_validation_confirmation_pass,...,redundancy_result,incremental_information_result,economic_interpretation,authorization_gate_open,authorization_gate_reason,pre_authorization_verdict,verdict,final_decision,verdict_reason,garch_audit_verdict
0,bollinger_zscore_20,MLAT-H001,4 / Appendix,131-133; 740-742,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,15/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,ECONOMICALLY_TRIVIAL,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...,REJECTED_IMPLEMENTATION
1,bollinger_bandwidth_20,MLAT-H002,Appendix,740-742,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,20/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...,REJECTED_IMPLEMENTATION
2,cutler_rsi_14,MLAT-H003,4,132,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,17/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,ECONOMICALLY_TRIVIAL,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...,REJECTED_IMPLEMENTATION
3,chaikin_money_flow_20,MLAT-H004,Appendix,752-753,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,11/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,ECONOMICALLY_TRIVIAL,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...,REJECTED_IMPLEMENTATION
4,amihud_illiquidity_60,MLAT-H005,20 / Appendix,656; 752,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,19/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...,REJECTED_IMPLEMENTATION
5,parkinson_volatility_30,MLAT-H006,9,297-301,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,24/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...,REJECTED_IMPLEMENTATION
6,rogers_satchell_volatility_30,MLAT-H007,9,297-301,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,22/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...,REJECTED_IMPLEMENTATION
7,realized_semivariance_balance_60,MLAT-H008,5 / 9,169-178; 297-301,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,15/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 horizon-thinning gate is structurall...,REJECTED_IMPLEMENTATION
8,bipower_jump_ratio_60,MLAT-H009,9,297-301,True,False,<NA>,True,True,True,...,PASS: no exact or >=0.995 existing-feature ove...,10/32 60/180 horizon-control tests pass,"No incremental, stable, nonredundant economic ...",False,frozen v1 horizon-thinning gate is structurall...,WEAK_OR_UNSTABLE,RESEARCH_ONLY,RESEARCH_ONLY,frozen v1 

### 13.2 Limitations

In [66]:
LIMITATIONS = [
    "Historical Final-test outcomes were previously exposed and remain excluded.",
    "MGC transfer validation was not authorized and was not inspected.",
    "Minute rows are dependent; dates, sessions, thinning, and block uncertainty are used.",
    ADVANCEMENT_GATE_REASON,
    "No sequential signal, transaction-cost model, PnL, or Sharpe ratio is produced.",
    "Any surviving feature still requires a new future holdout or live paper period.",
]
LIMITATIONS

['Historical Final-test outcomes were previously exposed and remain excluded.',
 'MGC transfer validation was not authorized and was not inspected.',
 'Minute rows are dependent; dates, sessions, thinning, and block uncertainty are used.',
 'frozen v1 horizon-thinning gate is structurally non-evaluable: 0/384 required 60/180-minute feature-family-session-partition cells have finite thinned daily-IC evidence under the >=10 observations-per-New-York-date rule, including 0 Development cells; advancement is not authorized',
 'No sequential signal, transaction-cost model, PnL, or Sharpe ratio is produced.',
 'Any surviving feature still requires a new future holdout or live paper period.']

### 13.3 Saved Outputs

In [67]:
evidence_tables = {
    "feature_observation_audit": build_result.observation_audit,
    "daily_ic_evidence": cell_results.daily_ic,
    "incremental_daily_partial_ic": incremental_result.daily_partial_ic,
}
saved_data = {
    "feature_registry": str(registry_path.relative_to(ROOT)),
    "feature_matrix": str(feature_path.relative_to(ROOT)),
}
artifact_verification = [
    {
        "artifact": "feature_registry",
        "format": "parquet",
        "checks": len(registry_save_checks),
        "passed": bool(registry_save_checks["passed"].all()),
    },
    {
        "artifact": "feature_matrix",
        "format": "parquet",
        "checks": len(feature_save_checks) + len(reload_checks),
        "passed": bool(
            feature_save_checks["passed"].all()
            and reload_checks["passed"].all()
        ),
    },
]
for name, table in evidence_tables.items():
    path = PATHS.data_path(name)
    entry = save_parquet(
        table,
        path,
        manifest_path=PATHS.manifest_path,
    )
    checks = verify_saved_table(
        table,
        path,
        manifest_entry=entry,
    )
    artifact_verification.append({
        "artifact": name,
        "format": "parquet",
        "checks": len(checks),
        "passed": bool(checks["passed"].all()),
    })
    saved_data[name] = str(path.relative_to(ROOT))

figure_manifest = pd.DataFrame([
    {
        "figure_name": path.stem,
        "path": str(path.relative_to(ROOT)),
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
    }
    for path in sorted(PATHS.figure_dir.glob("*.png"))
])
assert len(figure_manifest) >= 4
garch_warning_table = pd.DataFrame({
    "warning": [str(item.message) for item in garch_warnings]
})
tables_to_save = {
    "construction_performance": construction_performance,
    "feature_build_timings": build_result.build_timings,
    "feature_build_runtime": build_result.runtime_summary,
    "feature_construction_audit": build_result.construction_audit,
    "feature_engineering_diagnostics": build_result.feature_diagnostics,
    "feature_persistence_validation": build_result.persistence_validation,
    "feature_summary": feature_summary.reset_index(names="feature_name"),
    "upstream_integrity_checks": upstream_checks,
    "registry_save_checks": registry_save_checks,
    "feature_save_checks": feature_save_checks,
    "feature_reload_checks": reload_checks,
    "validation_checks": validation_result.checks,
    "validation_feature_diagnostics": validation_result.feature_diagnostics,
    "validation_coverage": validation_result.coverage,
    "validation_sample_hash": validation_result.sample_hash,
    "evaluation_checks": evaluation_result.checks,
    "evaluation_target_catalog": evaluation_result.target_catalog,
    "evaluation_family_coverage": evaluation_result.family_coverage,
    "univariate_cells": cell_results.cell_results,
    "quintile_edges": cell_results.quintile_edges,
    "quintile_results": cell_results.quintile_results,
    "year_stability": cell_results.year_stability,
    "thinning_results": cell_results.thinning_results,
    "overlap_audit": overlap_result,
    "candidate_correlation": candidate_correlation.reset_index(
        names="feature_name"
    ),
    "incremental_summary": incremental_result.summary,
    "feature_verdicts": final_verdicts,
    "garch_parameters": garch_audit.parameters,
    "garch_fit_diagnostics": garch_audit.fit.diagnostics,
    "garch_residual_diagnostics": garch_audit.residual_diagnostics,
    "garch_calibration": garch_audit.calibration,
    "garch_benchmarks": garch_audit.benchmark_metrics,
    "garch_audit_checks": garch_audit.audit_checks,
    "garch_warnings": garch_warning_table,
    "figure_manifest": figure_manifest,
}
saved_tables = {}
for name, table in tables_to_save.items():
    table = prepare_csv_table(table)
    path = PATHS.table_path(name)
    entry = save_csv(
        table,
        path,
        manifest_path=PATHS.manifest_path,
    )
    checks = verify_saved_table(
        table,
        path,
        manifest_entry=entry,
    )
    artifact_verification.append(
        {
            "artifact": name,
            "format": "csv",
            "checks": len(checks),
            "passed": bool(checks["passed"].all()),
        }
    )
    saved_tables[name] = str(path.relative_to(ROOT))

artifact_verification = pd.DataFrame(artifact_verification)
assert artifact_verification["passed"].all()
verification_path = PATHS.table_path("artifact_verification")
verification_entry = save_csv(
    artifact_verification,
    verification_path,
    manifest_path=PATHS.manifest_path,
)
verification_checks = verify_saved_table(
    artifact_verification,
    verification_path,
    manifest_entry=verification_entry,
)
assert verification_checks["passed"].all()
saved_tables["artifact_verification"] = str(
    verification_path.relative_to(ROOT)
)
artifact_manifest = pd.DataFrame(
    json.loads(PATHS.manifest_path.read_text(encoding="utf-8"))["artifacts"].values()
)
artifact_verification, artifact_manifest

(                           artifact   format  checks  passed
 0                  feature_registry  parquet      15    True
 1                    feature_matrix  parquet      26    True
 2         feature_observation_audit  parquet      15    True
 3                 daily_ic_evidence  parquet      15    True
 4      incremental_daily_partial_ic  parquet      15    True
 5          construction_performance      csv      15    True
 6             feature_build_timings      csv      15    True
 7             feature_build_runtime      csv      15    True
 8        feature_construction_audit      csv      15    True
 9   feature_engineering_diagnostics      csv      15    True
 10   feature_persistence_validation      csv      15    True
 11                  feature_summary      csv      15    True
 12        upstream_integrity_checks      csv      15    True
 13             registry_save_checks      csv      15    True
 14              feature_save_checks      csv      15    True
 15     

### 13.4 Recommended Next Research Stage

In [68]:
NEXT_STAGE = (
    "Freeze a chronological regularized-linear benchmark on the surviving shortlist."
    if MULTIVARIATE_AUTHORIZED
    else (
        "Preregister an MLAT v2 contract with a feasible non-overlap "
        "sensitivity statistic before examining new outcomes; retain v1 "
        "as fail-closed RESEARCH_ONLY evidence and require a new future "
        "holdout or live paper period."
    )
)
NEXT_STAGE

'Preregister an MLAT v2 contract with a feasible non-overlap sensitivity statistic before examining new outcomes; retain v1 as fail-closed RESEARCH_ONLY evidence and require a new future holdout or live paper period.'

### 13.5 Completion Gate

In [69]:
observed_partitions = set(
    evaluation_frame["research_partition"].dropna().astype(str)
)
DEVELOPMENT_VALIDATION_ONLY = observed_partitions == set(PARTITIONS)
FINAL_OUTCOMES_LOADED = bool(
    labels["research_partition"].astype(str).str.contains(
        "Final", case=False, regex=False
    ).any()
)
MGC_LOADED = bool(
    not bars["product"].eq("GC").all()
    or not labels["product"].eq("GC").all()
    or not existing_features["product"].eq("GC").all()
)
EVALUATION_GATE = bool(evaluation_result.checks["passed"].all())
ARTIFACT_GATE = bool(artifact_verification["passed"].all())
UPSTREAM_GATE = bool(
    upstream_checks[[
        "exists", "row_count_pass", "column_count_pass", "sha256_pass",
        "schema_fingerprint_recorded",
    ]].all().all()
)
execution_manifest = {
    **RUNTIME_IDENTITY,
    "run_seconds": time.perf_counter() - RUN_STARTED,
    "rss_bytes_final": PROCESS.memory_info().rss,
    "engineering_gate": ENGINEERING_GATE,
    "evaluation_gate": EVALUATION_GATE,
    "advancement_gate_feasible": ADVANCEMENT_GATE_FEASIBLE,
    "advancement_gate_reason": ADVANCEMENT_GATE_REASON,
    "artifact_gate": ARTIFACT_GATE,
    "upstream_gate": UPSTREAM_GATE,
    "development_validation_only": DEVELOPMENT_VALIDATION_ONLY,
    "final_outcomes_loaded": FINAL_OUTCOMES_LOADED,
    "mgc_loaded": MGC_LOADED,
    "multivariate_authorized": MULTIVARIATE_AUTHORIZED,
    "garch_verdict": GARCH_VERDICT,
    "garch_warnings": [str(item.message) for item in garch_warnings],
    "batch_sha256": BATCH_SHA256,
    "contract_sha256": CONTRACT_SHA256,
    "saved_data": saved_data,
    "saved_tables": saved_tables,
    "saved_figures": figure_manifest["path"].tolist(),
    "next_stage": NEXT_STAGE,
}
execution_path = PATHS.json_path("execution_manifest")
save_json(
    execution_manifest,
    execution_path,
    manifest_path=PATHS.manifest_path,
)
assert json.loads(execution_path.read_text(encoding="utf-8")) == execution_manifest
assert execution_manifest["engineering_gate"]
assert execution_manifest["evaluation_gate"]
assert not execution_manifest["advancement_gate_feasible"]
assert execution_manifest["artifact_gate"]
assert execution_manifest["upstream_gate"]
assert execution_manifest["development_validation_only"]
assert not execution_manifest["final_outcomes_loaded"]
assert not execution_manifest["mgc_loaded"]
execution_manifest

{'python': '3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]',
 'platform': 'Windows-11-10.0.26200-SP0',
 'git_commit': 'e75cab144f9f2096d51262c5da478166e116c9c5',
 'git_branch': 'twatski',
 'seed': 20260723,
 'dependencies': {'numpy': '2.2.3',
  'pandas': '3.0.3',
  'pyarrow': '25.0.0',
  'scipy': '1.17.1',
  'matplotlib': '3.11.1',
  'nbformat': '5.10.4'},
 'run_seconds': 2159.152393199998,
 'rss_bytes_final': 1726779392,
 'engineering_gate': True,
 'evaluation_gate': True,
 'advancement_gate_feasible': False,
 'advancement_gate_reason': 'frozen v1 horizon-thinning gate is structurally non-evaluable: 0/384 required 60/180-minute feature-family-session-partition cells have finite thinned daily-IC evidence under the >=10 observations-per-New-York-date rule, including 0 Development cells; advancement is not authorized',
 'artifact_gate': True,
 'upstream_gate': True,
 'development_validation_only': True,
 'final_outcomes_loaded': False,
 'mgc_loaded': Fal